# Cluster 1: Grammar-Constrained Compile Acceptance

This notebook reports Cluster 1 compile acceptance only. The scope is the frozen grammar-v1 ReLU/Softmax/GEMM experiment, represented here as `elementwise`, `reduction`, and `matmul` kernel families.

The results do not evaluate numerical correctness, runtime performance, speedup, memory safety, or generalization beyond the scoped kernel families.

## Load Artifacts

Load the frozen combined artifact: `outputs/cluster1/final_none_vs_g_l4_n20.jsonl`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "cluster1").exists():
            return candidate
    raise RuntimeError("could not locate TritonGen repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cluster1.experiments.make_cluster1_figures import (  # noqa: E402
    CONDITIONS,
    DTYPES,
    FIGURE_DIR,
    KERNEL_CLASSES,
    load_generation_results,
    make_tables,
    results_to_dataframe,
    validate_integrity,
    write_all_figures,
)


pd.set_option("display.max_columns", 40)
ARTIFACT_PATH = REPO_ROOT / "outputs/cluster1/final_none_vs_g_l4_n20.jsonl"
FIGURES = REPO_ROOT / "outputs/cluster1/figures"

rows = load_generation_results(ARTIFACT_PATH)
df = results_to_dataframe(rows)

{
    "artifact": str(ARTIFACT_PATH.relative_to(REPO_ROOT)),
    "rows": len(df),
    "columns": list(df.columns),
}

## Integrity Checks

The checks assert the frozen two-condition, three-kernel-family, three-dtype, n=20 design and grammar-specific masked-token-rate invariants.

In [ ]:
validate_integrity(df)

cell_counts = (
    df.groupby(["condition", "kernel_class", "dtype"], observed=False)
    .size()
    .rename("n")
    .reset_index()
)

checks = {
    "total_rows": len(df),
    "condition_counts": df.groupby("condition", observed=False).size().to_dict(),
    "kernel_classes": list(df["kernel_class"].cat.categories),
    "dtypes": list(df["dtype"].cat.categories),
    "cell_n_values": sorted(cell_counts["n"].unique().tolist()),
    "baseline_masked_token_rate_null": bool(
        df.loc[df["condition"].astype(str) == "baseline", "masked_token_rate"].isna().all()
    ),
    "g_masked_token_rate_non_null": bool(
        df.loc[df["condition"].astype(str) == "G", "masked_token_rate"].notna().all()
    ),
    "duplicate_condition_kernel_dtype_seed_identities": int(
        df.duplicated(["condition", "kernel_class", "dtype", "generation_seed"]).sum()
    ),
}
checks

## Summary Tables

In [ ]:
tables = make_tables(df)
figure_paths = write_all_figures(df, tables, FIGURES)
{name: str(path.relative_to(REPO_ROOT)) for name, path in figure_paths.items()}

### Headline Compile@1

In [ ]:
tables["headline"]

### Per-Kernel Compile@1

In [ ]:
tables["per_kernel"]

### Per-Dtype Compile@1

In [ ]:
tables["per_dtype"]

## Plot 1 - Headline Compile@1

In [ ]:
display(Image(filename=str(figure_paths["headline"])))

## Plot 2 - Compile@1 by Kernel Family

In [ ]:
display(Image(filename=str(figure_paths["by_kernel"])))

## Plot 3 - Failure Type Distribution

Baseline rows should show `SignatureError`; G rows should show `None/success`.

In [ ]:
tables["failure"]

In [ ]:
display(Image(filename=str(figure_paths["failure"])))

## Plot 4 - Masked Token Rate for G

`masked_token_rate` is a constraint-strength diagnostic, not quality/performance.

In [ ]:
tables["masked_token"]

In [ ]:
display(Image(filename=str(figure_paths["masked_token"])))

## Plot 5 - Diversity / Unique Solution Rate

Tight grammar may reduce diversity; interpret alongside compile acceptance.

In [ ]:
tables["diversity"]

In [ ]:
display(Image(filename=str(figure_paths["diversity"])))

## Optional Plot 6 - Compile-Only pass@k

The JSONL rows do not contain precomputed pass@k fields, so this notebook computes compile-only pass@1/pass@5/pass@10 from strict `compile_success` using the shared pass@k implementation.

In [ ]:
tables["pass_at_k"]

In [ ]:
display(Image(filename=str(figure_paths["pass_at_k"])))

## Final Interpretation

G improved compile acceptance from 0/180 to 180/180 under the strict Cluster 1 compile gate. The result supports grammar constraints as a structural validity mechanism for this scoped ReLU/Softmax/GEMM setting.

This result does not imply numerical correctness or speedup. Future clusters handle correctness, feedback, and performance.